# <center> <img src="../notebooks/img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Pipeline de Streaming — NYC Taxi Trips** </center>
---
**Profesor**: Pablo Camarillo Ramirez  
**Alumno**: Emilio Daniel Guzmán Seda

## Introducción

### Descripción de los Datos

Este pipeline procesa registros sintéticos de viajes de taxi de la ciudad de Nueva York (NYC Taxi Trips). Cada registro contiene **19 campos** que incluyen información temporal (pickup/dropoff), geográfica (LocationIDs), financiera (fare, tips, tolls) y operativa (VendorID, payment_type). El productor Kafka inyecta **datos sucios** de forma controlada:
- ~2% de registros con `trip_distance` negativa o cero
- ~1% de registros con `fare_amount` negativa o cero
- ~2% de registros con `PULocationID` o `DOLocationID` nulos
- `congestion_surcharge` y `airport_fee` siempre `None`
- ~1% de registros duplicados

### Diagrama de Arquitectura

```
┌─────────────────────┐         ┌─────────────────────┐         ┌─────────────────────┐
│                     │         │                     │         │                     │
│  kafka_producer.py  │────────▶│   Apache Kafka      │────────▶│  consumer.ipynb     │
│                     │  JSON   │   (kafka:9093)      │  Stream │  (PySpark SS)       │
│  - Genera viajes    │  UTF-8  │   Topic: taxi-trips │         │                     │
│  - Datos sucios     │         │                     │         │  foreachBatch:      │
│  - while True loop  │         └─────────────────────┘         │  ┌───────────────┐  │
│                     │                                         │  │ 1. Dedup      │  │
└─────────────────────┘                                         │  │ 2. Clean      │  │
                                                                │  │ 3. Join Zones │  │
                                ┌─────────────────────┐         │  │ 4. Aggregate  │  │
                                │                     │         │  │ 5. Write Neo4j│  │
                                │  taxi+_zone_lookup  │────────▶│  └───────────────┘  │
                                │  .csv (Static DF)   │  Join   │                     │
                                │  data/taxi_zones/   │         └──────────┬──────────┘
                                └─────────────────────┘                    │
                                                                           │ Neo4j Connector
                                                                           ▼
                                                                ┌─────────────────────┐
                                                                │                     │
                                                                │  Neo4j              │
                                                                │  (bolt://neo4j-     │
                                                                │   iteso:7687)       │
                                                                │                     │
                                                                │  (:Borough)         │
                                                                │  (:Trip)            │
                                                                │  [:PICKUP_IN]       │
                                                                │                     │
                                                                └─────────────────────┘
```

### Justificación Tecnológica

- **Apache Kafka**: Plataforma de mensajería distribuida que permite desacoplar el productor del consumidor. Ofrece durabilidad, escalabilidad horizontal y garantías de entrega at-least-once. Ideal para ingestión de datos en tiempo real.
- **Spark Structured Streaming**: Motor de procesamiento de streams que unifica batch y streaming bajo la misma API de DataFrames. Permite aplicar transformaciones complejas (joins, agregaciones, filtros) con semántica exactly-once mediante checkpoints.
- **Neo4j**: Base de datos de grafos nativa que permite modelar relaciones entre entidades (viajes y boroughs) de forma natural. El conector Spark-Neo4j facilita la escritura directa desde DataFrames usando Cypher.

In [1]:
# Instalar dependencias si es necesario
!pip install pyspark kafka-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 1.6 MB/s eta 0:00:0000:0100:02
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 KB 1.1 MB/s eta 0:00:00a 0:00:01
  Created wheel for pyspark: filename=pyspark-4.1.1-py2.py3-none-any.whl size=456008663 sha256=e248b2d942d10a1f36e2e2ee2973381e2a8f6387a422d9fa04a4261c46065a1f
  Stored in directory: /root/.cache/pip/wheels/16/77/d3/d15aaaab1df8384ad9bd94caba26a1a5aa439d8afd187a5ab9
Successfully built pyspark


In [ ]:
# Imports y creación de SparkSession
from spark_utils import SparkUtils
from pyspark.sql.functions import col, avg, sum, from_json
from pyspark.sql.types import StructType

# Paquetes necesarios: Kafka connector + Neo4j connector
packages = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0,org.neo4j:neo4j-connector-apache-spark_2.13:5.3.10_for_spark_3"

su = SparkUtils(
    app_name="NYC_Taxi_Streaming_Pipeline",
    master_url="spark://spark-master:7077",
    spark_packages=packages
)

su._spark

In [ ]:
# Definición del esquema de los 19 campos del taxi NYC
columns_info = [
    ("VendorID", "long"),
    ("tpep_pickup_datetime", "timestamp"),
    ("tpep_dropoff_datetime", "timestamp"),
    ("passenger_count", "long"),
    ("trip_distance", "double"),
    ("RatecodeID", "long"),
    ("store_and_fwd_flag", "string"),
    ("PULocationID", "long"),
    ("DOLocationID", "long"),
    ("payment_type", "long"),
    ("fare_amount", "double"),
    ("extra", "double"),
    ("mta_tax", "double"),
    ("tip_amount", "double"),
    ("tolls_amount", "double"),
    ("improvement_surcharge", "double"),
    ("total_amount", "double"),
    ("congestion_surcharge", "double"),
    ("airport_fee", "double")
]

# Generar esquema StructType a partir de columns_info
taxi_schema = SparkUtils.generate_schema(columns_info)
taxi_schema

In [ ]:
# Cargar DataFrame estático de zonas de taxi
zones_df = su._spark.read.csv(
    "/opt/spark/work-dir/data/taxi_zones/taxi+_zone_lookup.csv",
    header=True,
    inferSchema=True
)

zones_df.show(5)
zones_df.printSchema()

## Fase 1: Ingestión de Datos desde Kafka

In [ ]:
# Lectura del stream desde Kafka y parseo de JSON
kafka_df = (su._spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9093")
    .option("subscribe", "taxi-trips")
    .load())

# Convertir el valor binario de Kafka a string JSON
json_df = kafka_df.selectExpr("CAST(value AS STRING) as json_str")

# Parsear el JSON usando el esquema definido
df_parsed = (json_df
    .select(from_json(col("json_str"), taxi_schema).alias("data"))
    .select("data.*"))

df_parsed.printSchema()

## Fase 2: Función foreachBatch — Limpieza, Enriquecimiento y Persistencia

In [ ]:
# Configuración de conexión a Neo4j
neo4j_url = "bolt://neo4j-iteso:7687"
neo4j_user = "neo4j"
neo4j_passwd = "neo4j@1234"

neo4j_options = {
    "url": neo4j_url,
    "authentication.basic.username": neo4j_user,
    "authentication.basic.password": neo4j_passwd,
    "batch.size": "5000",
    "transaction.retries": "3"
}


def process_micro_batch(batch_df, batch_id):
    """Procesa cada micro-batch: limpieza, enriquecimiento, agregación y escritura a Neo4j."""
    # 1. Verificar batch vacío
    if batch_df.isEmpty():
        print(f"Batch {batch_id}: vacío, saltando...")
        return

    print(f"--- Batch {batch_id}: {batch_df.count()} registros recibidos ---")

    # 2. Deduplicación
    df_clean = batch_df.dropDuplicates()

    # 3. Eliminar columnas innecesarias
    df_clean = df_clean.drop("congestion_surcharge", "airport_fee")

    # 4. Eliminar nulos en campos críticos
    df_clean = df_clean.dropna(subset=["PULocationID", "DOLocationID", "fare_amount"])

    # 5. Filtrar registros inválidos
    df_clean = df_clean.filter((col("trip_distance") > 0) & (col("fare_amount") > 0))

    if df_clean.isEmpty():
        print(f"Batch {batch_id}: sin registros válidos después de limpieza")
        return

    # 6. Stream-Static Join con zonas
    df_enriched = df_clean.join(zones_df, df_clean.PULocationID == zones_df.LocationID, "left") \
                          .withColumnRenamed("Borough", "Pickup_Borough") \
                          .drop("LocationID", "Zone", "service_zone")

    # 7. Filtrar registros sin borough (no match en join)
    df_with_borough = df_enriched.filter(col("Pickup_Borough").isNotNull())

    # 8. Escribir nodos Borough en Neo4j
    borough_df = df_with_borough.select(col("Pickup_Borough").alias("name")).distinct()
    borough_df.write \
        .format("org.neo4j.spark.DataSource") \
        .mode("Overwrite") \
        .options(**neo4j_options) \
        .option("labels", ":Borough") \
        .option("node.keys", "name") \
        .save()

    # 9. Escribir nodos Trip en Neo4j
    trip_df = df_with_borough.select(
        col("VendorID").alias("vendor_id"),
        col("tpep_pickup_datetime").cast("string").alias("pickup_dt"),
        col("tpep_dropoff_datetime").cast("string").alias("dropoff_dt"),
        "passenger_count", "trip_distance", "fare_amount",
        "tip_amount", "total_amount", "payment_type",
        col("Pickup_Borough").alias("pickup_borough")
    )
    trip_df.write \
        .format("org.neo4j.spark.DataSource") \
        .mode("Append") \
        .options(**neo4j_options) \
        .option("labels", ":Trip") \
        .option("node.keys", "vendor_id,pickup_dt,pickup_borough") \
        .save()

    # 10. Escribir relaciones PICKUP_IN con query Cypher
    rel_query = """
    MATCH (t:Trip {vendor_id: event.vendor_id, pickup_dt: event.pickup_dt, pickup_borough: event.pickup_borough})
    MATCH (b:Borough {name: event.pickup_borough})
    MERGE (t)-[:PICKUP_IN]->(b)
    """
    trip_df.write \
        .format("org.neo4j.spark.DataSource") \
        .mode("Append") \
        .options(**neo4j_options) \
        .option("query", rel_query) \
        .save()

    # 11. Calcular y mostrar agregaciones
    df_agg = df_with_borough.groupBy("Pickup_Borough", "VendorID", "payment_type") \
        .agg(
            avg("tip_amount").alias("avg_tip"),
            sum("total_amount").alias("total_revenue")
        )
    print(f"Batch {batch_id}: Agregaciones calculadas")
    df_agg.show(10, truncate=False)

## Fase 3: Ejecución del Stream

In [ ]:
from pathlib import Path
import shutil

# Limpieza previa del checkpoint (patrón de clase)
checkpoint_path = "/opt/spark/work-dir/checkpoints/taxi_streaming_checkpoint"
dir_path = Path(checkpoint_path)
if dir_path.exists() and dir_path.is_dir():
    shutil.rmtree(dir_path)

# Iniciar el stream con foreachBatch
query = (df_parsed.writeStream
    .foreachBatch(process_micro_batch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(processingTime="10 seconds")
    .start())

print("Pipeline de streaming iniciado. Presiona Ctrl+C para detener.\n")
su._spark.streams.awaitAnyTermination()

In [ ]:
su.spark.stop()